In [ ]:
from src.preprocess.parquet_preprocessor import ParquetPreprocessor

ParquetPreprocessor.csv_to_parquet("data/raw/active_alarms_prod.csv", "data/raw/active_alarms_prod.parquet")

In [1]:
import polars as pl
from dotenv import load_dotenv
import os

load_dotenv(".env")

from src.repository.alarm_graph_repository import AlarmGraphRepository

lazy_frame = pl.scan_parquet("data/raw/active_alarms_prod.parquet")
graph_repo = AlarmGraphRepository(os.getenv("ACTIVE_DB_PATH"))

In [2]:
from src.pipelines.simple_time_correlation import SimpleTimeCorrelationActive

SimpleTimeCorrelationActive.train(lazy_frame, graph_repo)

In [3]:
from src.postprocess.enumerate_incidents import EnumerateIncidents

EnumerateIncidents.enumerate_data(graph_repo)

graph_repo.preview_nodes()

alert_id,incident,alert_type,start_time,end_time,node_id
str,i32,str,datetime[μs],datetime[μs],str
"""227d1b9c-fce4-4f15-a1c2-cc530c…",0,"""CONNECTIVITY_PORT_STATE_INFO""",2025-11-08 21:20:17.162669,2025-12-05 18:25:14.397934,"""4c98ad29-ce57-42a7-9fdb-f46d87…"
"""bc5fbc90-51c6-4e5e-9ecb-0264d1…",0,"""CONNECTIVITY_PORT_STATE_SFP_AM…",2025-08-22 18:41:09.054549,2025-12-05 17:42:12.543613,"""4c98ad29-ce57-42a7-9fdb-f46d87…"
"""2201ee65-d32a-41d9-af75-b2d653…",0,"""CONNECTIVITY_PORT_STATE_INFO""",2025-11-08 21:20:17.162669,2025-12-05 18:25:14.397934,"""4c98ad29-ce57-42a7-9fdb-f46d87…"
"""fdfb2827-2ed4-487b-a59c-9439c3…",0,"""CONNECTIVITY_PORT_STATE_SFP_AM…",2025-08-22 18:41:09.054549,2025-12-05 17:42:12.543613,"""4c98ad29-ce57-42a7-9fdb-f46d87…"
"""59787d84-d058-41a0-9c50-b8fa33…",0,"""HARDWARE_INTEGRITY_FAN_STATUS_…",2025-08-22 19:00:29.219868,2025-12-05 18:00:19.282548,"""4c98ad29-ce57-42a7-9fdb-f46d87…"
…,…,…,…,…,…
"""979f496f-5d6e-480d-b9bf-c9d92a…",0,"""CONNECTIVITY_PORT_STATE_INFO""",2025-11-08 21:20:17.162669,2025-12-05 18:25:14.397934,"""4c98ad29-ce57-42a7-9fdb-f46d87…"
"""d2fe184b-270d-43b4-baa9-7016fc…",0,"""CONNECTIVITY_PORT_STATE_SFP_AM…",2025-08-22 18:41:09.054549,2025-12-05 17:42:12.543613,"""4c98ad29-ce57-42a7-9fdb-f46d87…"
"""25ba2b1a-e8e5-4be7-bba1-cb9898…",0,"""CONNECTIVITY_PORT_STATE_INFO""",2025-11-08 21:20:17.162669,2025-12-05 18:25:14.397934,"""4c98ad29-ce57-42a7-9fdb-f46d87…"


In [4]:
from src.utils.node_summary import node_summary
from src.repository.aggregate_results_repository import AggregateResultsRepository

summary, general_metrics = node_summary(graph_repo)

results_repo = AggregateResultsRepository(filename="simple_time_corr_active")
results_repo.save(summary)
results_repo.load()

✅ Nova versão salva com sucesso em: data/results/simple_time_corr_active_20260528_204146.csv
📖 Carregando a versão mais recente encontrada: simple_time_corr_active_20260528_204146.csv


Node ID,Total de Alarmes,Total de Correlações,Total de Incidentes,Média de Alarmes por Incidente,Densidade
str,i64,i64,i64,f64,f64
"""c60cc1af-aa30-491a-a3ef-55c6f8…",399,417,130,3.069231,0.005252
"""f3ba8721-2bb0-46b3-b4dd-c9255b…",237,12358,18,13.166667,0.441894
"""4c43b243-0774-4296-8f18-85c699…",57,4,4,14.25,0.002506
"""4a278aa6-9cfe-4d85-ae05-9843e5…",60,4,4,15.0,0.00226
"""f02a8eab-0a93-4f87-90f9-3eb96a…",60,4,4,15.0,0.00226
…,…,…,…,…,…
"""d17942ed-da6b-4379-86cd-c23908…",1,0,1,1.0,0.0
"""770bd060-95c8-45a3-85c5-6134dd…",1,0,1,1.0,0.0
"""e3e55b67-76d8-4e6c-8928-265981…",1,0,1,1.0,0.0


In [5]:
graph_repo.close()